In [1]:
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment

c:\SEM 5\LAB AI Semester 5\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


# 1. Load data

In [2]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = "speech.wav"
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

# 2. Export Function

In [3]:
def export_pydub(audio):
    while True:
        file_name = input("Input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("File name must ends with .wav")
    audio.export(file_name, format="wav")
    print("Audio saved as: ", file_name)

def export_librosa(audio):
    while True:
        file_name = input("Input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("File name must ends with .wav")
    sf.write(file_name, np.clip(audio, -1, 1), sr)
    print("Audio saved as: ", file_name)

# 3. Predict Text Wav2Vec

In [4]:
def predict_text(audio_file):
    audio_np, _ = librosa.load(audio_file, sr=bundle.sample_rate, mono=True)
    waveform = torch.from_numpy(audio_np).unsqueeze(0)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

# 4. Augmentation

In [5]:
# augmentasi ada 9
# 1. adjvolume
def adjvolume(clip):
    while True:
        nilai = float(input("input nilai volume (-10000 to 100): "))
        if not (-10000 <= nilai <= 100):
            print("Nilai must be between -10000 to 100.")
        else:
            break
    export_pydub(clip + nilai)

# 2. repeat audio
def repeat_audio(clip):
    while True:
        nilai = int(input("input nilai volume (1 - 5): "))
        if not (1 <= nilai <= 5):
            print("Nilai must be between 1 to 5.")
        else:
            break
    export_pydub(clip * nilai)

# 3. fade in 
# def fade_in(clip):
#     while True:
#         nilai = float(input("input nilai greater than 0: "))
#         if nilai <= 0:
#             print("Nilai must be greater than 0.")
#         else:
#             break
#     export_pydub(clip.fade_in(nilai * 1000))

import random
def fade_in(clip):
    nilai = random.randint(1, 5)
    export_pydub(clip.fade_in(nilai * 1000)) # ini karena diminta di second jadi dikali 1000, kalo ms gausah dikali lagi

# 4. fade out
def fade_out(clip):
    nilai = random.randint(1, 5)
    export_pydub(clip.fade_out(nilai * 1000))

# 5. reverse audio
def reverse_audio(clip):
    export_pydub(clip.reverse())

# 6. adjpitch
def adjpitch():
    while True:
        nilai = float(input("input pitch rate (-10 - 10): "))
        if not (-10 <= nilai <= 10):
            print("Nilai must be between -10 to 10.")
        else:
            break
    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=nilai)
    export_librosa(shifted_audio)

# 7. adjspeed
def adjspeed():
    while True:
        nilai = float(input("input nilai speed (1 - 5): "))
        if not (1 <= nilai <= 5):
            print("Nilai must be between 1 to 5.")
        else:
            break
    speed_audio = librosa.effects.time_stretch(audio, rate=nilai)
    export_librosa(speed_audio)

# 8. white noise
def white_noise():
    while True:
        nilai = float(input("input nilai volume (1 - 5): "))
        if not (1 <= nilai <= 5):
            print("Nilai must be between 1 to 5.")
        else:
            break
    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * nilai)

# 9. random gain
def random_gain():
    nilai = np.random.uniform(0.5, 1.5)
    print(f"Random Gain x {nilai:.2f}")
    export_librosa(audio * nilai)

# 5. Aug Menu

In [6]:
def aug_menu():
    while True:
        print("1. Increase or decrease volume")
        print("2. Increase or decrease pitch")
        print("3. Insert fade in effect ")
        print("4. Insert fade out effect")
        print("5. Back to main menu")

        choice = input(">> ")
        if choice == "1":
            adjvolume(clip)
        elif choice == "2":
            adjpitch()
        elif choice == "3":
            fade_in(clip)
        elif choice == "4":
            fade_out(clip)
        elif choice == "5":
            break
        else:
            print("Invalid input")


# 6. Main Menu

In [7]:
def main_menu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")

        choice = input(">> ")
        if choice == "1":
            print("Audio Augmentation Now")
            aug_menu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Hasil Transcript")
            print(transcript)
        elif choice == "3":
            break
        else:
            print("Invalid input")
    print("Main Menu Closed")

main_menu()

1. Audio Augmentation
2. Predict Text
3. Exit
Hasil Transcript
I REALLY LOVE TITE SO I LIKE TO HUG DETE 
1. Audio Augmentation
2. Predict Text
3. Exit
Hasil Transcript
I REALLY LOVE TITE SO I LIKE TO HUG DETE 
1. Audio Augmentation
2. Predict Text
3. Exit
Main Menu Closed
